# Capítulo 9 — IA Embarcada e Computação em Tempo Real

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Compreender** as restrições das plataformas embarcadas — **energia, memória, latência e computação** — e como elas redesenham o projeto de um modelo para o campo;
2. **Entender** a *edge computing* e a operação em **ambientes negados** de conectividade, e por que a autonomia em relação à nuvem é um requisito operacional e de soberania;
3. **Aplicar** técnicas de otimização de modelos — **quantização**, ***pruning*** e **destilação** — trocando acurácia por eficiência de forma controlada e medida;
4. **Implantar** modelos com **TensorFlow Lite** e **ONNX** e empregar aceleradores dedicados;
5. **Projetar** uma cadeia de inferência embarcada para uma plataforma real (VANT, veículo terrestre não tripulado), equilibrando **acurácia, latência, energia e confiabilidade**.

> Os Módulos I a III construíram modelos que percebem, decidem e resistem a ataques — sempre, porém, sobre um pressuposto implícito de **computação abundante**: o treino em GPUs, a inferência em um *notebook*. **O campo é outro mundo.** Ali, um modelo precisa correr em um VANT, em um equipamento portátil, em um veículo terrestre não tripulado — plataformas com limites severos de energia, memória e latência, e frequentemente **desconectadas de qualquer nuvem**.
>
> A lição que organiza o capítulo é simples e severa: **o melhor modelo que não cabe na plataforma é inútil.** A implantação não é um detalhe posterior à modelagem; é um problema de **engenharia de primeira classe**, que muitas vezes deve guiar o projeto do modelo desde o início.

## Preparação do ambiente

O capítulo emprega **duas cadeias de ferramentas**, refletindo as listagens do livro — cada qual usa a ferramenta certa para a sua tarefa:

- **`TensorFlow` / TF Lite** para a **quantização** (Listagem 9.1) — o caminho canônico de implantação embarcada em Keras;
- **`PyTorch` + `ONNX`** para *pruning*, destilação, exportação e medição de latência (Listagens 9.2–9.4).

`tensorflow` e `torch` já vêm no Colab; `onnx` e `onnxruntime` são instalados em célula própria. Tudo roda em CPU em poucos minutos. Como sempre, fixamos a **semente aleatória**.

In [ ]:
# Preparacao do ambiente: importacoes e sementes de reprodutibilidade
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEMENTE = 0
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)

import io, contextlib
@contextlib.contextmanager
def silencioso():
    # suprime as mensagens internas do exportador do TF Lite ("Saved
    # artifact at ...") -- emitidas tanto por print() Python quanto pelo
    # file descriptor C++ -- mantendo a saida da celula limpa.
    lixo = io.StringIO()
    with open(os.devnull, "w") as devnull:
        fd_out, fd_err = os.dup(1), os.dup(2)
        os.dup2(devnull.fileno(), 1); os.dup2(devnull.fileno(), 2)
        try:
            with contextlib.redirect_stdout(lixo), contextlib.redirect_stderr(lixo):
                yield
        finally:
            os.dup2(fd_out, 1); os.dup2(fd_err, 2)
            os.close(fd_out); os.close(fd_err)

print(f"numpy   {np.__version__}")
print(f"pytorch {torch.__version__}")
print("Ambiente pronto.")

---
## 9.1 As restrições da borda

Entre o laboratório e o campo há um **abismo de recursos**. O desenvolvimento supõe computação farta; a implantação embarcada, o oposto. Quatro restrições, entrelaçadas, governam esse novo regime:

- A **energia** é a mais implacável em plataformas a bateria: cada inferência consome potência, e **potência gasta em computação é autonomia e alcance perdidos**;
- A **memória** (RAM e armazenamento) é limitada — um modelo de centenas de megabytes simplesmente não cabe;
- A **latência** é imposta pelo tempo real: uma detecção que chega tarde demais é inútil, por mais correta que seja (Capítulo 4);
- A **computação disponível** é uma fração ínfima da de um centro de dados — sem falar da dissipação **térmica**.

O efeito é uma **mudança nas métricas que importam**. Não basta a acurácia; passa a importar a **acurácia por watt, por milissegundo e por megabyte**. O projeto embarcado vive sobre uma *fronteira de eficiência*, em que cada ganho de qualidade é pesado contra o seu custo em energia, tempo e espaço.

> **🛡️ Contexto de defesa** — Em um VANT, cada *watt* gasto em inferência é autonomia de voo perdida, e cada milissegundo de latência é uma decisão desatualizada. **O modelo não pode ser escolhido à parte da plataforma: os dois são coprojetados.** Um detector que exige mais computação do que o VANT carrega, ou que responde mais devagar do que o fluxo de quadros exige, não é um bom modelo com um problema de engenharia — é um **modelo inadequado à missão**. A restrição da borda não é um obstáculo a contornar; é **parte da especificação**.

Para trabalhar, treinamos um **modelo "de laboratório"** — grande e acurado, no espírito do professor que comprimiremos ao longo do capítulo — sobre um conjunto sintético de **assinaturas de sensor** $20 \times 20$ em seis classes:

In [ ]:
# Conjunto sintetico de assinaturas de sensor: 6 classes 20x20
LADO, N_CLASSES = 20, 6
NOMES = ["barra |", "barra --", "ponto", "anel", "diagonal", "L"]

def gera_dados(n_por_classe, ruido=0.09, semente=SEMENTE):
    rng = np.random.default_rng(semente)
    X, y = [], []
    for c in range(N_CLASSES):
        for _ in range(n_por_classe):
            img = rng.normal(0.12, ruido, (LADO, LADO))
            cx, cy = rng.integers(5, 15, 2)
            if c == 0:                                  # barra vertical
                img[cy - 4:cy + 4, cx] += 0.6
            elif c == 1:                                # barra horizontal
                img[cy, cx - 4:cx + 4] += 0.6
            elif c == 2:                                # ponto
                img[cy - 1:cy + 1, cx - 1:cx + 1] += 0.7
            elif c == 3:                                # anel
                yy, xx = np.ogrid[:LADO, :LADO]
                img[np.abs((xx - cx) ** 2 + (yy - cy) ** 2 - 9) < 3] += 0.55
            elif c == 4:                                # diagonal
                for k in range(-3, 4):
                    if 0 <= cy + k < LADO and 0 <= cx + k < LADO:
                        img[cy + k, cx + k] += 0.6
            else:                                       # forma em L
                img[cy - 3:cy + 3, cx] += 0.55
                img[cy + 2, cx:cx + 3] += 0.55
            X.append(np.clip(img, 0, 1))
            y.append(c)
    X = np.array(X, dtype=np.float32)[:, None, :, :]    # (n, 1, 20, 20)
    y = np.array(y, dtype=np.int64)
    ordem = rng.permutation(len(X))
    return X[ordem], y[ordem]

X_tr, y_tr = gera_dados(300)
X_te, y_te = gera_dados(300, semente=99)

class Professor(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128), nn.ReLU(),
            nn.Linear(128, N_CLASSES),
        )

    def forward(self, x):
        return self.rede(x)

def treina(modelo, X, y, epocas=40, lote=64, prof=None, T=4.0, alfa=0.4,
           semente=SEMENTE):
    modelo.train()
    otim = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    Xt, yt = torch.tensor(X), torch.tensor(y)
    rng = np.random.default_rng(semente)
    for _ in range(epocas):
        for i in rng.permutation(len(Xt))[:len(Xt) // lote * lote].reshape(-1, lote):
            idx = torch.tensor(i)
            logits = modelo(Xt[idx])
            if prof is None:
                perda = F.cross_entropy(logits, yt[idx])
            else:                                       # destilacao (Sec. 9.3.3)
                with torch.no_grad():
                    lp = prof(Xt[idx])
                perda = perda_destilacao(logits, lp, yt[idx], T, alfa)
            otim.zero_grad(); perda.backward(); otim.step()
    return modelo

def acuracia(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        pred = modelo(torch.tensor(X)).argmax(1).numpy()
    return (pred == y).mean()

def n_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

torch.manual_seed(SEMENTE)
professor = treina(Professor(), X_tr, y_tr)
print(f"acuracia do professor: {acuracia(professor, X_te, y_te):.3f}")
print(f"parametros:            {n_parametros(professor):,}")
print(f"tamanho em float32:    {n_parametros(professor) * 4 / 1024:.0f} KB")

**Observe:** um modelo acurado — e "pesado" para uma plataforma de campo. É esse modelo que as técnicas do capítulo vão **encolher e acelerar** até caber no orçamento da borda.

## 9.2 Edge computing e ambientes negados

Por que processar **na própria plataforma** (na *borda*), em vez de enviar os dados a um servidor remoto? Em defesa, quatro razões o impõem:

1. **Latência** — a ida e volta até uma nuvem é lenta demais para decisões em tempo real;
2. **Conectividade** — o campo de batalha é um ambiente **DDIL**: conectividade *Negada, Degradada, Intermitente e Limitada*. O adversário **bloqueia e degrada deliberadamente** as comunicações; não se pode pressupor um enlace com a retaguarda;
3. **Segurança e soberania** — enviar dados de sensor a uma nuvem remota — possivelmente estrangeira — é um risco de *opsec* e de soberania (Capítulos 6 e 8);
4. **Autonomia** — a plataforma precisa funcionar **quando cortada de tudo**.

A **edge computing** é a resposta: **trazer a inferência para a plataforma**. A implicação de projeto é direta — o modelo deve ser autossuficiente e eficiente o bastante para correr localmente, sem apoio externo. É essa exigência que torna as técnicas de otimização da próxima seção **não um luxo, mas uma necessidade**.

> **📝 Nota** — A autonomia na borda tem uma dimensão de **soberania** que percorre todo o livro. Um sistema que depende de uma nuvem estrangeira para *pensar* é um sistema que o detentor dessa nuvem pode **desligar, observar ou degradar**. Operar na borda, com modelos abertos e auto-hospedados (Capítulo 6), sobre *hardware* sob controle nacional, é a materialização, no plano técnico, do princípio de **soberania tecnológica** do Capítulo 1. Em ambiente negado, essa autonomia deixa de ser preferência e torna-se **a diferença entre operar e ficar cego**.

## 9.3 Otimização de modelos: quantização, *pruning* e destilação

Fazer um modelo caber e correr na borda exige **encolhê-lo e acelerá-lo**. Três técnicas, frequentemente combinadas, dominam o campo — todas trocando alguma acurácia por eficiência, na mesma lógica de **compromisso medido** que o livro defende.

### 9.3.1 Quantização

A **quantização** reduz a precisão numérica dos pesos e das ativações — tipicamente de **ponto flutuante de 32 bits para inteiros de 8 bits**. O mapeamento é simples: um valor real $r$ é representado por um inteiro $q$,

$$q = \mathrm{round}\!\left(\frac{r}{s}\right) + z, \qquad r \approx s\,(q - z),$$

onde $s$ é um **fator de escala** e $z$ o **ponto zero**. O ganho é substancial: um modelo em *int8* ocupa cerca de **um quarto** do espaço de um em *float32*, executa aritmética inteira mais rápida e consome menos energia. O custo é uma **pequena perda de acurácia** — controlável pela *quantização consciente do treino*.

A célula seguinte instala as bibliotecas de implantação; em seguida, a **Listagem 9.1** faz a quantização pós-treino com o TensorFlow Lite:

In [ ]:
# Bibliotecas de implantacao (no Colab, ~30 s). tensorflow ja vem.
%pip install -q onnx onnxruntime

In [ ]:
# Listagem 9.1 - Quantizacao pos-treino com TensorFlow Lite (float32 -> int8).
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# O TF Lite parte de um modelo Keras. Treinamos aqui um equivalente ao
# professor (mesmo dado, layout NHWC) para ilustrar a quantizacao.
Xk_tr = np.transpose(X_tr, (0, 2, 3, 1))     # (n, 20, 20, 1) para o Keras
Xk_te = np.transpose(X_te, (0, 2, 3, 1))
keras.utils.set_random_seed(SEMENTE)
modelo_keras = keras.Sequential([
    layers.Input((LADO, LADO, 1)),
    layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(128, activation="relu"),
    layers.Dense(N_CLASSES),
])
modelo_keras.compile(optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"])
modelo_keras.fit(Xk_tr, y_tr, epochs=8, batch_size=128, verbose=0)

# Converte o modelo Keras (float32) em um TFLite quantizado (int8):
# cerca de 4x menor, mais rapido e de menor consumo.
conversor = tf.lite.TFLiteConverter.from_keras_model(modelo_keras)
conversor.optimizations = [tf.lite.Optimize.DEFAULT]

# Um conjunto representativo calibra as faixas de quantizacao.
def dados_representativos():
    for x in Xk_tr[:200]:
        yield [x[None].astype("float32")]

conversor.representative_dataset = dados_representativos
conversor.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conversor.inference_input_type = tf.int8
conversor.inference_output_type = tf.int8
with silencioso():
    modelo_int8 = conversor.convert()
    tflite_f32 = tf.lite.TFLiteConverter.from_keras_model(modelo_keras).convert()
with open("modelo_int8.tflite", "wb") as f:
    f.write(modelo_int8)

# tamanho float32 (referencia) vs int8, para a comparacao
print(f"TFLite float32: {len(tflite_f32) / 1024:6.1f} KB")
print(f"TFLite int8:    {len(modelo_int8) / 1024:6.1f} KB")
print(f"compressao:     {len(tflite_f32) / len(modelo_int8):.2f}x")

In [ ]:
# Aferir SEMPRE a acuracia do modelo quantizado: a compressao tem um
# custo que precisa ser medido no alvo, nunca presumido.
def acuracia_tflite(conteudo_tflite, X_nhwc, y):
    intr = tf.lite.Interpreter(model_content=conteudo_tflite)
    intr.allocate_tensors()
    ent = intr.get_input_details()[0]
    sai = intr.get_output_details()[0]
    s, z = ent["quantization"]                 # escala e ponto-zero (int8)
    acertos = 0
    for i in range(len(X_nhwc)):
        q = np.round(X_nhwc[i] / s + z).astype(np.int8)[None]
        intr.set_tensor(ent["index"], q)
        intr.invoke()
        if intr.get_tensor(sai["index"])[0].argmax() == y[i]:
            acertos += 1
    return acertos / len(y)

acc_f32 = modelo_keras.evaluate(Xk_te, y_te, verbose=0)[1]
acc_i8 = acuracia_tflite(modelo_int8, Xk_te, y_te)
print(f"acuracia float32: {acc_f32:.3f}")
print(f"acuracia int8:    {acc_i8:.3f}")
print(f"custo da quantizacao: {acc_i8 - acc_f32:+.3f} de acuracia por ~4x menos espaco")

**Observe:** o modelo *int8* ocupa cerca de **um quarto** do espaço, com perda de acurácia mínima — a troca favorável que torna a quantização o primeiro passo de quase toda implantação embarcada. Note a disciplina: a acurácia quantizada foi **medida**, não presumida (o Exercício 9.1 explora essa troca).

### 9.3.2 Pruning

O ***pruning*** (poda) remove os pesos que **pouco contribuem**. Muitos parâmetros de uma rede treinada são redundantes: zerá-los quase não afeta a saída. O *pruning* **não estruturado** zera pesos individuais, produzindo uma rede **esparsa**; o **estruturado** remove estruturas inteiras — canais, filtros, neurônios —, produzindo um modelo menor e **denso**, que acelera em qualquer *hardware*. Tipicamente, poda-se **de forma gradual**, com re-treino entre as etapas para recuperar a acurácia perdida.

A célula abaixo poda o professor em intensidades crescentes e mede o efeito — revelando **quanta redundância** a rede carrega, e onde está o limite:

In [ ]:
# Pruning nao estruturado global: quanta redundancia a rede carrega?
import torch.nn.utils.prune as prune

fracoes = [0.0, 0.5, 0.8, 0.9, 0.95]
acc_poda, esparsidades = [], []
for fracao in fracoes:
    m = Professor()
    m.load_state_dict(professor.state_dict())     # parte do professor treinado
    camadas = [mod for mod in m.modules()
               if isinstance(mod, (nn.Conv2d, nn.Linear))]
    if fracao > 0:
        # zera a fracao dos pesos de MENOR magnitude, globalmente
        prune.global_unstructured([(c, "weight") for c in camadas],
                                  pruning_method=prune.L1Unstructured,
                                  amount=fracao)
    zeros = sum(float((c.weight == 0).sum()) for c in camadas)
    total = sum(c.weight.numel() for c in camadas)
    esparsidades.append(zeros / total)
    acc_poda.append(acuracia(m, X_te, y_te))

fig, eixo = plt.subplots(figsize=(7.5, 4.6))
eixo.plot([f * 100 for f in fracoes], acc_poda, "o-", lw=2)
eixo.axhline(1 / N_CLASSES, color="gray", ls=":", label="acaso (6 classes)")
eixo.set_xlabel("pesos podados (%)"); eixo.set_ylabel("acuracia")
eixo.set_title("Pruning: muita redundancia ate um limiar, depois o colapso")
eixo.legend(); plt.tight_layout(); plt.show()

for f, e, a in zip(fracoes, esparsidades, acc_poda):
    print(f"poda {f:4.0%} | esparsidade real {e:4.0%} | acuracia {a:.3f}")

**Observe:** a rede sustenta a acurácia até **~80% dos pesos podados** — prova de quanta **redundância** ela carrega — e então **colapsa** (na poda de 90%, cai ao acaso). O ponto de colapso é o que dita quanto se pode comprimir; em produção, o re-treino entre etapas empurra esse limite adiante (Exercício 9.5).

### 9.3.3 Destilação de conhecimento

A **destilação** treina um modelo pequeno (o **aluno**) para imitar um grande (o **professor**), usando as **saídas suavizadas** deste como alvo. A função de perda combina o acerto nos rótulos verdadeiros com a aproximação das distribuições do professor, suavizadas por uma **temperatura** $T$:

$$\mathcal{L} = \alpha\, \mathcal{L}_{\mathrm{CE}}(\text{aluno}, y) + (1 - \alpha)\, T^2\, \mathrm{KL}\!\left(\sigma(z_{\mathrm{prof}}/T) \,\|\, \sigma(z_{\mathrm{aluno}}/T)\right),$$

onde $\sigma$ é o *softmax* e KL a divergência de Kullback–Leibler. O aluno resultante é **compacto** — cabe na plataforma — mas **retém grande parte da capacidade** do professor, porque aprende não só o rótulo certo, mas a *estrutura das confianças* do professor entre as classes. É o mesmo mecanismo da destilação defensiva do Capítulo 8, aqui a serviço da eficiência.

A **Listagem 9.2** mostra a função de perda:

In [ ]:
# Listagem 9.2 - Destilacao de conhecimento: a perda aluno-professor.
def perda_destilacao(logits_aluno, logits_professor, y, T=4.0, alfa=0.3):
    # O aluno (pequeno) imita o professor (grande) por suas saidas
    # SUAVIZADAS pela temperatura T.
    # 1. termo supervisionado: acerto nos rotulos verdadeiros
    perda_dura = F.cross_entropy(logits_aluno, y)
    # 2. termo de destilacao: aproxima as distribuicoes suavizadas
    p_prof = F.softmax(logits_professor / T, dim=1)
    log_p_aluno = F.log_softmax(logits_aluno / T, dim=1)
    perda_mole = F.kl_div(log_p_aluno, p_prof,
                          reduction="batchmean") * (T * T)
    return alfa * perda_dura + (1.0 - alfa) * perda_mole

# Aluno compacto: uma unica camada convolucional, ~4x menos parametros.
class Aluno(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(16 * 10 * 10, 32), nn.ReLU(),
            nn.Linear(32, N_CLASSES),
        )

    def forward(self, x):
        return self.rede(x)

# Treina o aluno com o laco padrao (Cap. 3), trocando apenas a perda.
torch.manual_seed(1)
aluno = treina(Aluno(), X_tr, y_tr, prof=professor, T=4.0, alfa=0.4)

acc_prof = acuracia(professor, X_te, y_te)
acc_aluno = acuracia(aluno, X_te, y_te)
print(f"professor: acc={acc_prof:.3f}  params={n_parametros(professor):,}")
print(f"aluno:     acc={acc_aluno:.3f}  params={n_parametros(aluno):,}")
print(f"-> {n_parametros(professor) / n_parametros(aluno):.0f}x menos parametros, "
      f"retendo {acc_aluno / acc_prof:.0%} da capacidade")

**Observe:** o aluno, com **cerca de 1/4 dos parâmetros** do professor, retém quase toda a sua acurácia — um modelo que agora **cabe na plataforma**. (A comparação com um aluno treinado *sem* destilação — Exercício 9.4 — traz uma lição sutil: nem sempre a destilação supera o treino direto; o ganho depende da estrutura do problema. É a regra do capítulo: **medir, não presumir**.)

> **✅ Boa prática** — As três técnicas comprimem, mas **alteram** o modelo — e um modelo alterado é um modelo diferente, que precisa ser **reavaliado**. A acurácia da versão otimizada deve ser medida no alvo, sob os mesmos critérios do original, **incluindo a robustez adversarial** (Capítulo 8): a quantização, por exemplo, pode afetar a robustez de maneiras não óbvias. A regra é a mesma de todo o livro — **medir, não presumir** —, agora aplicada à troca entre eficiência e qualidade.

## 9.4 Implantação: TensorFlow Lite, ONNX e aceleradores

Otimizado o modelo, resta **implantá-lo**. Três elementos compõem a cadeia:

- O **TensorFlow Lite** executa modelos TensorFlow/Keras em dispositivos móveis e embarcados — de *smartphones* a microcontroladores —, com interpretadores otimizados e suporte à quantização (Listagem 9.1);
- O **ONNX** (*Open Neural Network Exchange*) é um **formato interoperável**: treina-se em qualquer biblioteca (PyTorch, TensorFlow), exporta-se para ONNX e executa-se com o **ONNX Runtime** em uma grande variedade de alvos. Ele **desacopla o framework de treino do alvo de implantação** — vantagem de portabilidade e de **soberania**, por não prender a operação a um único fornecedor;
- Os **aceleradores dedicados** — GPUs embarcadas, **NPUs**, *edge* **TPUs** e **FPGAs** — fornecem o *hardware* especializado que executa a inferência com eficiência muito superior à de uma CPU genérica.

A **Listagem 9.3** exporta um modelo para ONNX e o executa **sem depender do PyTorch**:

In [ ]:
# Listagem 9.3 - Exportacao para ONNX e inferencia com o ONNX Runtime.
import onnxruntime as ort

# 1. Exporta o modelo PyTorch (o aluno compacto) para o formato ONNX.
professor.eval()
exemplo = torch.randn(1, 1, LADO, LADO)
torch.onnx.export(aluno, exemplo, "modelo.onnx",
                  input_names=["entrada"], output_names=["saida"],
                  dynamic_axes={"entrada": {0: "lote"}}, dynamo=False)

# 2. Executa a inferencia com o ONNX Runtime, SEM depender do PyTorch.
sessao = ort.InferenceSession("modelo.onnx",
                              providers=["CPUExecutionProvider"])
entrada = X_te[:1].astype("float32")
saida = sessao.run(None, {"entrada": entrada})[0]

# coerencia: a saida do ONNX Runtime coincide com a do PyTorch?
with torch.no_grad():
    saida_pt = aluno(torch.tensor(entrada)).numpy()
print(f"saida ONNX == saida PyTorch? diferenca max = "
      f"{np.abs(saida - saida_pt).max():.2e}")
print(f"classe prevista: {NOMES[saida.argmax()]}")
# O ONNX desacopla o framework de treino do alvo de implantacao --
# portabilidade e soberania: nao ficar preso a um unico fornecedor.

**Observe:** as duas saídas coincidem a menos de erro numérico ($\sim 10^{-6}$): o modelo exportado é **fielmente o mesmo**, agora num formato que roda em qualquer alvo com ONNX Runtime — sem carregar o PyTorch junto (o Exercício 9.2 explora o valor disso para a soberania).

---
## 9.5 Estudo de caso: inferência em VANT e plataformas não tripuladas

Reunimos tudo em um problema real: pôr um detector (Capítulo 4) a correr, **em tempo real**, a bordo de um VANT ou de um veículo terrestre não tripulado — plataforma a bateria, com computação modesta e **sem enlace garantido**. O processo de projeto **inverte a ordem habitual**: começa-se pelo **orçamento da plataforma**.

1. Primeiro, fixam-se os **limites**: a latência máxima admitida pelo fluxo de quadros, a energia disponível, a memória do dispositivo;
2. Escolhe-se e **otimiza-se** um modelo para caber nesse orçamento — arquitetura eficiente, quantizada e podada, eventualmente destilada;
3. **Implanta-se** via TFLite ou ONNX sobre o acelerador da plataforma;
4. E, por fim — passo **inegociável** —, **mede-se no alvo**: latência, vazão, energia e acurácia, na própria plataforma, e não no computador de desenvolvimento.

A **Listagem 9.4** realiza essa medição:

In [ ]:
# Listagem 9.4 - Medicao no alvo: latencia e vazao de inferencia.
import time

# Benchmark NO ALVO: latencia e vazao da inferencia na plataforma.
# O numero que decide se o modelo cabe no orcamento de tempo real.
sessao = ort.InferenceSession("modelo.onnx",
                              providers=["CPUExecutionProvider"])
entrada = X_te[:1].astype("float32")

for _ in range(10):                     # aquecimento (nao medir a partida)
    sessao.run(None, {"entrada": entrada})

tempos = []
for _ in range(200):                    # medicao
    t0 = time.perf_counter()
    sessao.run(None, {"entrada": entrada})
    tempos.append(time.perf_counter() - t0)

tempos = np.array(tempos) * 1000.0      # milissegundos
print(f"Latencia mediana: {np.median(tempos):.2f} ms")
print(f"Latencia p95:     {np.percentile(tempos, 95):.2f} ms")
print(f"Vazao:            {1000.0 / np.mean(tempos):.0f} inferencias/s")

# A latencia P95 -- nao a media -- deve caber no orcamento de tempo
# real: e a cauda que determina se um quadro sera perdido.
ORCAMENTO_MS = 33.0                      # 30 quadros/s -> 33 ms por quadro
cabe = np.percentile(tempos, 95) <= ORCAMENTO_MS
print(f"\nOrcamento de tempo real (30 q/s = 33 ms): "
      f"{'CABE' if cabe else 'NAO cabe'} (pela p95)")

**Observe:** dois pontos coroam o estudo. O primeiro é **a cauda, de novo**: é a latência de **pior caso (p95)**, não a média, que determina se um quadro será perdido — a mesma lição dos Capítulos 7 e 8, agora em tempo de inferência. O segundo é a **degradação graciosa**: uma plataforma embarcada confiável deve **falhar em segurança** — reduzir a taxa de quadros, sinalizar baixa confiança, ceder o controle ao operador — em vez de travar ou de produzir saídas silenciosamente erradas. E, como em todo o livro, **a plataforma percebe e tria; a decisão que importa permanece humana** — tema que o Capítulo 10 enfrenta de frente.

---
## 9.6 O caminho à frente

Este capítulo tornou a IA **implantável onde a operação ocorre**: sob as restrições da borda, em ambientes negados, comprimida por quantização, *pruning* e destilação, e servida por TFLite, ONNX e aceleradores. Resta o último passo — e o que dá sentido a todos os anteriores.

O **Capítulo 10** fecha a obra com a **ética, a regulação e a IA em sistemas de armas autônomos**: os marcos internacionais emergentes (a CCW da ONU, os princípios da OTAN, o AI Act europeu), o princípio do **controle humano significativo**, a posição do Brasil e os instrumentos do Ministério da Defesa, o direito internacional humanitário aplicado a sistemas autônomos e a **governança técnica** — auditabilidade, rastreabilidade e *accountability*.

Não é coincidência que a obra termine ali. A capacidade de operar **na borda, com autonomia e desconexão**, que este capítulo entregou, é precisamente o que torna a questão do controle humano **urgente**: quanto mais autônoma e independente a plataforma, mais aguda a pergunta sobre **quem responde pelo que ela faz**. Toda a disciplina acumulada — comparar, medir a cauda, validar o envelope, manter o humano na decisão — converge, no capítulo final, do plano técnico para o **plano normativo**.

## 📋 Resumo do capítulo

- A implantação embarcada é governada por **quatro restrições entrelaçadas** — energia, memória, latência e computação (mais a térmica). As métricas mudam: importam **acurácia por watt, por milissegundo e por megabyte**. Modelo e plataforma são **coprojetados**.

- A **edge computing** traz a inferência à plataforma por quatro razões: latência, conectividade **DDIL** (negada, degradada, intermitente, limitada), segurança/soberania e autonomia. Em ambiente negado, processar na borda é **a diferença entre operar e ficar cego**.

- A **quantização** reduz a precisão numérica ($\approx 4\times$ menor em *int8*); o ***pruning*** remove pesos redundantes (esparso ou estruturado); a **destilação** treina um aluno compacto a partir de um professor, via saídas suavizadas por temperatura.

- A implantação usa **TensorFlow Lite** (embarcados, com quantização), **ONNX** (formato interoperável que desacopla treino de alvo — vantagem de soberania) e **aceleradores** dedicados (GPU, NPU, TPU, FPGA).

- No estudo de caso, o projeto **parte do orçamento da plataforma**, otimiza o modelo para caber e **mede no alvo**: a latência de **pior caso (p95)**, não a média, decide se um quadro se perde.

- Uma plataforma embarcada confiável deve **degradar graciosamente** e **falhar em segurança**; e, mais autônoma a operação na borda, mais urgente o **controle humano** sobre a decisão — o tema do Capítulo 10.

## ⚠️ Armadilhas comuns

1. **Otimizar só a acurácia, ignorando o orçamento da plataforma.** O melhor modelo que não cabe na energia, na memória ou na latência disponíveis é inútil. Coprojete modelo e plataforma desde o início, partindo do orçamento.

2. **Pressupor conectividade com a nuvem.** O ambiente operacional é DDIL: o adversário nega e degrada as comunicações. Projete para operar na borda, com autonomia e desconexão.

3. **Quantizar ou podar sem reavaliar.** Um modelo comprimido é um modelo diferente: meça sua acurácia — e sua **robustez adversarial** — no alvo, sob os critérios do original. A compressão tem um custo que não se presume.

4. **Medir no computador de desenvolvimento, não na plataforma.** A latência e o consumo no alvo podem ser muito diferentes dos do *laptop*. Faça o *benchmark* no *hardware* real de emprego.

5. **Reportar a latência média em vez da cauda.** É a latência de pior caso (p95) que determina se um quadro será perdido em tempo real. Dimensione o modelo à **cauda**, não à média.

6. **Não prever a degradação graciosa.** Sob sobrecarga ou baixa confiança, a plataforma deve **falhar em segurança** — reduzir a taxa, sinalizar, ceder ao operador —, jamais travar ou errar em silêncio (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 9.1** — Com base na Listagem 9.1, a célula abaixo compara o **tamanho** do arquivo `.tflite` *int8* ao do *float32* e mede a **acurácia** antes e depois. Execute-a e relate, em duas linhas, a **troca entre compressão e qualidade** observada.

In [ ]:
# Exercicio 9.1 - a troca compressao vs. qualidade da quantizacao
kb_f32 = len(tflite_f32) / 1024
kb_i8 = len(modelo_int8) / 1024
print(f"tamanho: float32 = {kb_f32:.1f} KB | int8 = {kb_i8:.1f} KB "
      f"({kb_f32 / kb_i8:.2f}x menor)")
print(f"acuracia: float32 = {acc_f32:.3f} | int8 = {acc_i8:.3f} "
      f"({acc_i8 - acc_f32:+.3f})")

# Sua resposta (2 linhas):
# A quantizacao trocou ... de espaco por ... de acuracia, o que ...
# em uma plataforma a bateria significa ...

**Exercício 9.2** — A célula abaixo (Listagem 9.3) verifica que a saída do **ONNX Runtime** coincide, a menos de erro numérico, com a do modelo original em PyTorch. Execute-a e explique, em uma frase, por que a **interoperabilidade** do ONNX é valiosa para a **soberania** na implantação.

In [ ]:
# Exercicio 9.2 - a fidelidade da exportacao ONNX (e o que ela habilita)
sessao_ex = ort.InferenceSession("modelo.onnx",
                                 providers=["CPUExecutionProvider"])
lote = X_te[:64].astype("float32")     # <--- ALTERE o tamanho do lote
saida_onnx = sessao_ex.run(None, {"entrada": lote})[0]
with torch.no_grad():
    saida_pt = aluno(torch.tensor(lote)).numpy()
concordam = (saida_onnx.argmax(1) == saida_pt.argmax(1)).mean()
print(f"diferenca numerica maxima: {np.abs(saida_onnx - saida_pt).max():.2e}")
print(f"classes previstas identicas: {concordam:.1%}")

# Sua resposta (1 frase):
# A interoperabilidade do ONNX apoia a soberania porque ...

**Exercício 9.3** — A célula abaixo (Listagem 9.4) mede a latência e compara a **mediana** à **p95**. Execute-a e explique, em duas linhas, por que a **p95** — e não a média — é o número relevante para um requisito de **tempo real**.

In [ ]:
# Exercicio 9.3 - mediana vs. p95: por que a cauda decide
sessao_ex = ort.InferenceSession("modelo.onnx",
                                 providers=["CPUExecutionProvider"])
entrada = X_te[:1].astype("float32")
for _ in range(10):
    sessao_ex.run(None, {"entrada": entrada})
tempos = np.array([(lambda t0: (sessao_ex.run(None, {"entrada": entrada}),
                                time.perf_counter() - t0)[1])(time.perf_counter())
                   for _ in range(300)]) * 1000

print(f"latencia mediana: {np.median(tempos):.2f} ms")
print(f"latencia media:   {np.mean(tempos):.2f} ms")
print(f"latencia p95:     {np.percentile(tempos, 95):.2f} ms")
print(f"latencia p99:     {np.percentile(tempos, 99):.2f} ms")
print(f"pior caso:        {tempos.max():.2f} ms")

# Sua resposta (2 linhas):
# A p95/p99 importa mais que a media porque, em tempo real, um unico
# quadro que estoura o orcamento significa ...

### Tático

**Exercício 9.4** — A célula abaixo destila um aluno (Listagem 9.2) e o compara a um aluno **idêntico treinado sem destilação** e ao **professor**. Compare acurácia e tamanho, e discuta o ganho — **ou a ausência dele**. *(Atenção: neste problema sintético, o ganho da destilação sobre o treino direto pode ser pequeno ou nulo. Isso não é um defeito do código — é a lição do capítulo: o efeito de cada técnica deve ser **medido**, não presumido. A destilação brilha quando o professor codifica relações ricas entre classes; num problema de classes bem separadas, há pouco "conhecimento" extra a transferir.)*

In [ ]:
# Exercicio 9.4 - destilacao vs. treino direto vs. professor
torch.manual_seed(1)
aluno_direto = treina(Aluno(), X_tr, y_tr)          # SEM destilacao
# 'aluno' (com destilacao) e 'professor' ja foram treinados acima

for nome, m in [("professor", professor),
                ("aluno destilado", aluno),
                ("aluno sem destilacao", aluno_direto)]:
    print(f"{nome:22s} acc={acuracia(m, X_te, y_te):.3f}  "
          f"params={n_parametros(m):>7,}")

T_EXP, ALFA_EXP = 4.0, 0.4      # <--- ALTERE T e alfa e observe o efeito
torch.manual_seed(1)
aluno_exp = treina(Aluno(), X_tr, y_tr, prof=professor, T=T_EXP, alfa=ALFA_EXP)
print(f"\naluno destilado (T={T_EXP}, alfa={ALFA_EXP}): "
      f"acc={acuracia(aluno_exp, X_te, y_te):.3f}")

# Sua resposta (poucas linhas):
# 1) O aluno compacto retem ...% da acuracia do professor com ...x menos
#    parametros -- o ganho central da destilacao e' a COMPRESSAO.
# 2) Frente ao treino direto, a destilacao aqui ... porque ...

**Exercício 9.5** *(dissertativo)* — Dado um orçamento de latência de tempo real (por exemplo, **33 ms por quadro**, para 30 quadros por segundo), parta de um modelo que **não** o cumpre e aplique otimizações (quantização e *pruning*) até enquadrá-lo, **medindo a acurácia a cada etapa**. Documente a troca e o ponto em que a perda de acurácia se torna **inaceitável**.

*Responda em uma célula de texto (ou combine as células de código acima).*

**Exercício 9.6** *(dissertativo)* — Projete a cadeia de inferência de um detector para um VANT com restrição de energia. Especifique a **arquitetura**, as **otimizações**, o **formato de implantação** (TFLite ou ONNX), o **acelerador** e o **plano de medição no alvo**. Justifique cada escolha em função das **quatro restrições da borda**.

### Estratégico

**Exercício 9.7** — Em um texto técnico (até três páginas), defenda a tese de que, em ambiente **DDIL**, a autonomia de inferência na borda é um **requisito operacional e de soberania** — não uma otimização de desempenho. Relacione o argumento à auto-hospedagem de modelos do Capítulo 6 e à soberania tecnológica do Capítulo 1.

**Exercício 9.8** — Projete um mecanismo completo de **degradação graciosa** para uma plataforma autônoma embarcada: especifique como o sistema deve se comportar sob **sobrecarga computacional**, **baixa confiança do modelo** e **perda de sensores**, sempre falhando em segurança e preservando o controle humano. Justifique cada decisão do ponto de vista de confiabilidade e do Capítulo 10.

**Exercício 9.9** — Tome um modelo e uma plataforma-alvo (real ou emulada) e conduza um **estudo completo de implantação embarcada**: aplique quantização, *pruning* e destilação, meça no alvo a latência (mediana e p95), a vazão e a acurácia, e **reavalie a robustez adversarial** da versão otimizada (Capítulo 8). Documente em um miniartigo de até três páginas as trocas, o **envelope operacional** resultante e uma recomendação de emprego.

---

*Fim do Capítulo 9 — no Capítulo 10, o fecho da obra: ética, regulação e o controle humano significativo sobre a IA em sistemas de defesa.*